In [2]:
# 1. Câu lệnh kết nối trực tiếp đến tài liệu trên Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Cài đặt các thư viện cần thiết cho việc xây dựng mô hình
!pip install transformers underthesea -q

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 77.3 MB/s eta 0:00:00


In [3]:
import pandas as pd

path = '/content/drive/MyDrive/BTL AI/data_segmented.csv'

df = pd.read_csv(path)

df.head()

,Unnamed: 0,index,Statement,Context,annotation_id,Topic,Author,Url,labels,Evidence,split,Statement_word_len,Statement_char_len,Evidence_word_len,Evidence_char_len,Context_word_len,Context_char_len,Statement_seg,Evidence_seg
0,0,3049,"Phó Thủ tướng Trần Hồng Hà thay mặt Chính phủ,...","(Chinhphu.vn) - Đây là mong muốn, gửi gắm của ...",18933775,Chính trị,Chính Phủ,https://baochinhphu.vn/lan-toa-nhung-gia-tri-v...,0,"Thay mặt Chính phủ, Thủ tướng Chính phủ, Phó T...",train,62,293,62,293,1763,8270,Phó Thủ_tướng Trần_Hồng_Hà thay_mặt Chính_phủ ...,"Thay_mặt Chính_phủ , Thủ_tướng Chính_phủ , Phó..."
1,1,6811,Hành vi của Tô Văn Hải là cho phép người khác ...,"Ngày 24/3, Cơ quan Cảnh sát điều tra Công an t...",19402113,PHÁP LUẬT,Tin Tức,https://baotintuc.vn/an-ninh-trat-tu/bat-tam-g...,0,Tô Văn Hải đã có hành vi cho phép người khác đ...,train,24,109,48,225,880,4007,Hành_vi của Tô_Văn_Hải là cho_phép người khác ...,Tô_Văn_Hải đã có hành_vi cho_phép người khác đ...
2,2,7270,SAWACO thông báo tạm ngưng cung cấp nước để th...,(PLO)- Theo Tổng Công ty Cấp nước Sài Gòn (SAW...,19807472,ĐÔ THỊ,Báo Pháp Luật HCM,https://plo.vn/9-quan-huyen-o-tphcm-se-bi-cup-...,1,SAWACO thông báo tạm ngưng cung cấp nước để th...,train,44,202,35,162,244,1119,SAWACO thông_báo tạm ngưng cung_cấp nước để th...,SAWACO thông_báo tạm ngưng cung_cấp nước để th...
3,3,7423,"CLB luôn chuẩn bị rất kỹ lưỡng, chỉn chu chươn...","Với khoảng 200 thành viên, UEF Warm Hugs Club ...",19725173,Giáo dục,Người lao động,https://nld.com.vn/giao-duc-khoa-hoc/hanh-trin...,2,Là chương trình lớn nhất của CLB trong năm nên...,train,55,248,47,204,789,3597,"CLB luôn chuẩn_bị rất kỹ_lưỡng , chỉn_chu chươ...",Là chương_trình lớn nhất của CLB trong năm nên...
4,4,6632,"ILA tiếp nhận và hỗ trợ học sinh miễn phí, Bé ...","Đi qua 2 năm dịch bệnh đầy thử thách, nhưng ch...",19021291,Giáo dục,Thanh Niên,https://thanhnien.vn/giao-duc-anh-ngu-phat-tri...,2,"Bé được tham gia kiểm tra trình độ đầu vào, tư...",train,38,166,30,130,1551,7107,"ILA tiếp_nhận và hỗ_trợ học_sinh miễn_phí , Bé...","Bé được tham_gia kiểm_tra trình_độ đầu_vào , t..."


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Khai báo tên mô hình PhoBERT gốc trên kho của Hugging Face
model_name = "vinai/phobert-base"

# 2. Tải bộ từ điển (Tokenizer) để dịch chữ tiếng Việt thành số cho AI hiểu
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3. Tải bộ não (Model) và cấu hình đầu ra có 3 nhãn (Đúng, Sai, Không đủ thông tin)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# 4. Kiểm tra xem Colab đã bật Card đồ họa (GPU) chưa để chạy cho bốc
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"✅ Đã tải xong PhoBERT! Hệ thống đang chạy trên: {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Đã tải xong PhoBERT! Hệ thống đang chạy trên: cuda


In [5]:
from torch.utils.data import Dataset, DataLoader
import torch

# 1. Tách dữ liệu Train (Huấn luyện) và Dev (Khảo sát) theo đánh dấu của Nguyên
df_train = df[df['split'] == 'train'].reset_index(drop=True)
df_val = df[df['split'] == 'dev'].reset_index(drop=True)

# 2. Xây dựng khuôn mẫu để gói cặp câu (Nhận định + Bằng chứng) lại với nhau
class ViFactCheckDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=256):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        statement = str(self.df.loc[index, 'Statement_seg'])
        evidence = str(self.df.loc[index, 'Evidence_seg'])
        label = int(self.df.loc[index, 'labels'])

        # Dùng Tokenizer đã tải ở bước trước để biến chữ thành mã số
        encoding = self.tokenizer(
            statement,
            evidence,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# 3. Tạo DataLoader băm dữ liệu thành từng gói (Mỗi gói 16 mẫu)
train_loader = DataLoader(ViFactCheckDataset(df_train, tokenizer), batch_size=16, shuffle=True)
val_loader = DataLoader(ViFactCheckDataset(df_val, tokenizer), batch_size=16)

print(f"✅ Đã đóng gói xong! Tập Train có {len(df_train)} mẫu, Tập Dev có {len(df_val)} mẫu.")

✅ Đã đóng gói xong! Tập Train có 5062 mẫu, Tập Dev có 723 mẫu.


In [6]:
# Cập nhật: Gọi AdamW trực tiếp từ thư viện torch.optim
from torch.optim import AdamW
import torch

# 1. Cấu hình bộ tối ưu hóa (Optimizer) - Bộ phận giúp AI biết cách tự sửa sai
optimizer = AdamW(model.parameters(), lr=2e-5)

epochs = 3
print("🚀 BẮT ĐẦU HUẤN LUYỆN PHOBERT (CHẠY 3 VÒNG) 🚀\n")

# 2. Vòng lặp huấn luyện chính
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for step, batch in enumerate(train_loader):
        # Bê các gói dữ liệu 16 câu ném vào GPU
        b_input_ids = batch['input_ids'].to(device)
        b_attention_mask = batch['attention_mask'].to(device)
        b_labels = batch['labels'].to(device)

        # Xóa trí nhớ về cái sai cũ
        optimizer.zero_grad()

        # Đưa dữ liệu qua bộ não AI để nó dự đoán
        outputs = model(input_ids=b_input_ids, attention_mask=b_attention_mask, labels=b_labels)

        # Tính toán mức độ sai lệch (Loss)
        loss = outputs.loss
        total_loss += loss.item()

        # Lan truyền ngược để AI tự điều chỉnh lại dây thần kinh (Cập nhật trọng số)
        loss.backward()
        optimizer.step()

        # Cứ chạy được 50 gói thì báo cáo tiến độ 1 lần cho đỡ sốt ruột
        if step % 50 == 0 and step > 0:
            print(f"  Đang chạy gói {step}/{len(train_loader)} - Độ sai lệch (Loss): {loss.item():.4f}")

    # Tổng kết sau mỗi vòng
    avg_train_loss = total_loss / len(train_loader)
    print(f"✅ Hết Epoch {epoch + 1}/{epochs} - Lỗi trung bình của vòng này: {avg_train_loss:.4f}\n")

# 3. Xuất file sản phẩm cuối cùng
torch.save(model.state_dict(), 'phobert_vifactcheck.bin')
print("🎉 THÀNH CÔNG RỰC RỠ! Đã xuất file sản phẩm phobert_vifactcheck.bin để bàn giao cho nhóm!")

🚀 BẮT ĐẦU HUẤN LUYỆN PHOBERT (CHẠY 3 VÒNG) 🚀



model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

  Đang chạy gói 50/317 - Độ sai lệch (Loss): 0.9990
  Đang chạy gói 100/317 - Độ sai lệch (Loss): 1.1629
  Đang chạy gói 150/317 - Độ sai lệch (Loss): 0.7790
  Đang chạy gói 200/317 - Độ sai lệch (Loss): 0.9132
  Đang chạy gói 250/317 - Độ sai lệch (Loss): 0.6015
  Đang chạy gói 300/317 - Độ sai lệch (Loss): 0.5439
✅ Hết Epoch 1/3 - Lỗi trung bình của vòng này: 0.8950

  Đang chạy gói 50/317 - Độ sai lệch (Loss): 0.6312
  Đang chạy gói 100/317 - Độ sai lệch (Loss): 0.6851
  Đang chạy gói 150/317 - Độ sai lệch (Loss): 0.3394
  Đang chạy gói 200/317 - Độ sai lệch (Loss): 0.6023
  Đang chạy gói 250/317 - Độ sai lệch (Loss): 0.6994
  Đang chạy gói 300/317 - Độ sai lệch (Loss): 0.6593
✅ Hết Epoch 2/3 - Lỗi trung bình của vòng này: 0.4945

  Đang chạy gói 50/317 - Độ sai lệch (Loss): 0.1972
  Đang chạy gói 100/317 - Độ sai lệch (Loss): 0.4770
  Đang chạy gói 150/317 - Độ sai lệch (Loss): 0.4793
  Đang chạy gói 200/317 - Độ sai lệch (Loss): 0.1970
  Đang chạy gói 250/317 - Độ sai lệch (Loss):